# 🚀 ViralClip AI - Whisper Large-V3 Kaggle Free GPU Server
1. Go to right panel: **Settings -> Accelerator -> GPU T4 x2**
2. Ensure **Settings -> Internet -> Internet On** is enabled.
3. Run the cell below, copy the public URL, and paste it into ViralClip AI!

In [ ]:
# =====================================================================
# ViralClip AI - Whisper Large-V3 Kaggle Free GPU Server (All-in-One)
# =====================================================================
import os, sys, shutil, tempfile, subprocess, time, re, socket, uuid, threading

# 1. Ensure modern PyAV, CUDA 12 runtime, and faster-whisper are installed and up to date
print("📦 Installing/verifying dependencies (CUDA 12, av>=11.0.0, faster-whisper, fastapi, uvicorn)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "nvidia-cublas-cu12",
    "nvidia-cudnn-cu12",
    "av>=11.0.0",
    "faster-whisper",
    "fastapi",
    "uvicorn",
    "python-multipart"
])

# 2. Configure Dynamic Linker & Pre-load CUDA 12 Shared Libraries for CTranslate2
import site, glob, ctypes

def setup_cuda_libraries():
    """Locates and loads nvidia-cublas-cu12 and nvidia-cudnn-cu12 into dynamic linker and memory."""
    site_dirs = site.getsitepackages()
    if hasattr(site, "getusersitepackages"):
        site_dirs.append(site.getusersitepackages())

    cuda_lib_dirs = []
    for s in site_dirs:
        for pkg in ["cublas", "cudnn"]:
            p = os.path.join(s, "nvidia", pkg, "lib")
            if os.path.isdir(p) and p not in cuda_lib_dirs:
                cuda_lib_dirs.append(p)

    if cuda_lib_dirs:
        current_ld = os.environ.get("LD_LIBRARY_PATH", "")
        new_ld = ":".join(cuda_lib_dirs) + (f":{current_ld}" if current_ld else "")
        os.environ["LD_LIBRARY_PATH"] = new_ld

    sys_lib_candidates = ["/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/lib64", "/usr/lib"]
    target_sys_lib = next((d for d in sys_lib_candidates if os.path.isdir(d) and os.access(d, os.W_OK)), None)

    for lib_dir in cuda_lib_dirs:
        for so_file in glob.glob(os.path.join(lib_dir, "*.so*")):
            so_name = os.path.basename(so_file)
            if target_sys_lib:
                dest = os.path.join(target_sys_lib, so_name)
                if not os.path.exists(dest):
                    try:
                        os.symlink(so_file, dest)
                    except Exception:
                        pass
            try:
                ctypes.CDLL(so_file, mode=ctypes.RTLD_GLOBAL)
            except Exception:
                pass

setup_cuda_libraries()

# Safeguard against PyAV metadata_errors keyword mismatch across all versions
try:
    import av
    _orig_av_open = av.open
    def _safe_av_open(*args, **kwargs):
        try:
            return _orig_av_open(*args, **kwargs)
        except TypeError as e:
            if "metadata_errors" in str(e):
                kwargs.pop("metadata_errors", None)
                return _orig_av_open(*args, **kwargs)
            raise
    av.open = _safe_av_open

    # Also patch inside faster_whisper audio module
    import faster_whisper.audio
    faster_whisper.audio.av.open = _safe_av_open
except Exception as patch_err:
    pass

# 3. Download cloudflared binary
if not os.path.exists("./cloudflared-linux-amd64"):
    print("📥 Downloading Cloudflare binary...")
    os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x cloudflared-linux-amd64")

from faster_whisper import WhisperModel
from fastapi import FastAPI, UploadFile, File
import uvicorn

app = FastAPI(title="Kaggle Whisper Large-V3 GPU Server")

# Check and Load Model on GPU
try:
    print("⚡ Loading Whisper Large-v3 on GPU (T4)...")
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")
    print("✅ Whisper Large-v3 Model Ready on GPU!")
except Exception as e:
    print(f"\n⚠️ GPU initialization notice: {e}")
    print("⚡ Falling back to CPU mode...")
    model = WhisperModel("medium", device="cpu", compute_type="int8")
    print("✅ Whisper Model Ready on CPU!")

jobs_db = {}

def run_transcription_worker(job_id: str, file_path: str):
    global model
    try:
        print(f"⚙️ [Job {job_id}] Transcribing (Fast Turbo Mode)...")
        try:
            segments, info = model.transcribe(
                file_path,
                beam_size=1,
                best_of=1,
                word_timestamps=True,
                vad_filter=True,
                condition_on_previous_text=False
            )
        except Exception as gpu_exec_err:
            print(f"⚠️ GPU transcription error: {gpu_exec_err}")
            print("⚡ Auto-recovering: Switching to CPU transcription...")
            cpu_model = WhisperModel("medium", device="cpu", compute_type="int8")
            segments, info = cpu_model.transcribe(
                file_path,
                beam_size=1,
                best_of=1,
                word_timestamps=True,
                vad_filter=True,
                condition_on_previous_text=False
            )

        result_segments = []
        all_words = []

        for seg in segments:
            words = []
            if getattr(seg, "words", None):
                for w in seg.words:
                    word_item = {
                        "word": w.word.strip(),
                        "start": round(w.start, 3),
                        "end": round(w.end, 3)
                    }
                    words.append(word_item)
                    all_words.append(word_item)

            result_segments.append({
                "id": seg.id,
                "start": round(seg.start, 3),
                "end": round(seg.end, 3),
                "text": seg.text.strip(),
                "words": words
            })

        jobs_db[job_id] = {
            "status": "completed",
            "result": {
                "status": "success",
                "language": getattr(info, "language", "en"),
                "duration": round(getattr(info, "duration", 0.0), 2),
                "segments": result_segments,
                "words": all_words
            }
        }
        print(f"✅ [Job {job_id}] Complete! Duration: {round(getattr(info, 'duration', 0.0), 2)}s, Segments: {len(result_segments)}")
    except Exception as e:
        print(f"❌ [Job {job_id}] Failed: {e}")
        jobs_db[job_id] = {"status": "error", "error": str(e)}
    finally:
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
            except Exception:
                pass

@app.get("/")
@app.get("/health")
def health_check():
    return {
        "status": "online",
        "service": "ViralClip AI Whisper Large-V3 Kaggle GPU Server",
        "gpu": True
    }

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    job_id = str(uuid.uuid4())[:8]
    print(f"📥 Received {file.filename} -> Assigned Job: {job_id}")
    ext = os.path.splitext(file.filename or "")[1] or ".mp3"
    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = tmp.name

    jobs_db[job_id] = {"status": "processing"}
    worker = threading.Thread(target=run_transcription_worker, args=(job_id, tmp_path), daemon=True)
    worker.start()

    return {
        "status": "processing",
        "job_id": job_id
    }

@app.get("/status/{job_id}")
def check_status(job_id: str):
    if job_id not in jobs_db:
        return {"status": "not_found"}
    return jobs_db[job_id]

# Cleanup old processes
os.system("pkill -9 -f cloudflared 2>/dev/null")
os.system("fuser -k 8000/tcp 2>/dev/null")

# Dynamically find free port
def get_free_port(preferred=8000):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(('127.0.0.1', preferred))
            return preferred
    except OSError:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(('127.0.0.1', 0))
            return s.getsockname()[1]

LOCAL_PORT = get_free_port(8000)
print(f"🔌 Server binding on local port: {LOCAL_PORT}")

# Launch Cloudflare Tunnel
print("🚀 Starting Cloudflare Tunnel (HTTP/2 Mode)...")
log_file = open("cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{LOCAL_PORT}"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

url = None
print("⏳ Waiting for Cloudflare Tunnel URL...")
for _ in range(30):
    time.sleep(1)
    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                url = match.group(0)
                break

if url:
    print("\n" + "="*65)
    print("🔥 YOUR PUBLIC KAGGLE WHISPER URL IS:")
    print(f"👉  {url}  👈")
    print("="*65)
    print("⚠️  KEEP THIS CELL RUNNING! Do NOT stop this cell.\n")
else:
    print("⚠️ Check cloudflared.log")

config = uvicorn.Config(app, host="127.0.0.1", port=LOCAL_PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()
